# Steam's Gaming Recommender Algorithm
*Reyna Geluk, Stan Bakker, Mieke Fraters & Aryan Müller*

Over here we can put some cute background info, and our goal for this project.

## Getting relevant data and cleaning datasets 
We put the links here and stuff, with argumentation as to why we chose/needed them.

### Steam reviews dataset
The dataset can be found [here.](https://www.kaggle.com/datasets/kieranpoc/steam-reviews/data)


The dataset by Kieranpo’c on Kaggle contains around 114 million rows, with each row consisting of users and their like/dislike labels for specific games. Whenever a user writes a review about a game they own, Steam asks them to say whether they liked and recommend a game. This data is stored in the column `voted_up` and contains `1` and `0`.

Although having this much data at our disposal is useful in theory, our hardware sadly does not have the power to run the entire dataset, especially when considering the two-week timeframe for this project. Therefore, the data has been filtered to only contain relevant entries and columns from users with a minimum of seven reviews. Although setting that limit at four would be most optimal to filter out the lowtail users, GitHub can’t handle those large files. Filtering like this lessens the amount of bias compared to other filtering methods without destroying system memory, while keeping this project feasible.


In [1]:
# Once im sure everything is ready and wont break this file and let it take ages to load, ill add my shabam of data cleaning and stuff in here.

In [2]:
#Imports
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [3]:

df = pd.read_csv("allFilteredAggr7.csv") # Change dataset depending on what files we use!!!!
df = df[["recommendationid", "appid", "author_steamid", "author_num_reviews", "voted_up"]]
quantile = 0.5 # Amount may differ according to our research results
df_filtered = df[df["author_num_reviews"] > df["author_num_reviews"].quantile(quantile)]
display(df_filtered)


,recommendationid,appid,author_steamid,author_num_reviews,voted_up
0,148919350,10,76561198826729322,21,1
1,148905699,10,76561198820122182,39,1
2,148898785,10,76561198853145449,60,1
3,148881526,10,76561199090138342,31,1
4,148865538,10,76561199015747395,34,1
...,...,...,...,...,...
48806036,50215500,997240,76561197988663774,347,1
48806037,50214389,997240,76561198272374716,3364,1
48806038,50215529,997250,76561197988663774,347,1
48806039,50215531,997251,76561197988663774,347,0


## Creating the recommender algoirhtm

In [6]:
# ═════════════════════════════════════════════════════════════════════════════
# 0. MY NOTES AND THOUGHTS
# ═════════════════════════════════════════════════════════════════════════════
#So I need to make:

# Collaborative filtering: Users like you also liked. Using players with similiar voting patterns and stuff. (User-User KNN)
# Content- based filtering: Games similar to ones you liked. We are gonna use the tags. (TF-IDF + Cosine Similarity)
# Hybrid recommender: combine the two from above
# I will just be figuring out what the dude did and copy it so that is why the code might be a bit weird? like wtf are all those cat things
# They make my head spin with the logic behind it XD oeh I am gonna use the fancy code barriers
# Oh yea I used printing inbetween to see what it was doing and if going wrong where


# ═════════════════════════════════════════════════════════════════════════════
# 1. MAKING THE CODE USABLE OR SOMETHING
# ═════════════════════════════════════════════════════════════════════════════

# Map users and apps to contiguous integer indices.
#So bassically what this part does. It grabs the information from the columns. Tells panda to see it as type "category" aka unique labels not just numbers
#And apparently cat.codes replaces the unique label with a small integer assigned in sorted order. So no more huge numbers? idk
# So .loc[:. "name"] is need or else pandas starts complaining "A value is trying to be set on a copy of a slice from a DataFrame." it legit told me
# To do it this way
df_filtered.loc[:, "user_idx"] = df_filtered["author_steamid"].astype("category").cat.codes
df_filtered.loc[:, "app_idx"]  = df_filtered["appid"].astype("category").cat.codes

# .cat.categories is the reverse lookup: a sorted array of the original IDs, where position i holds the original ID that was assigned cat.code i.
# We save this so we can convert back from matrix indices to real IDs later.
unique_user_ids = df_filtered["author_steamid"].astype("category").cat.categories  # index → steamid
unique_app_ids  = df_filtered["appid"].astype("category").cat.categories           # index → appid

n_users = df_filtered["user_idx"].max() + 1   # total rows in our matrix
n_apps  = df_filtered["app_idx"].max() + 1    # total columns in our matrix

#this is just a check
print(f"\nUnique users : {n_users:,}")
print(f"Unique apps  : {n_apps:,}")


# ═════════════════════════════════════════════════════════════════════════════
# 2. COLLABORATIVE FILTERING  (User-User KNN)
# ═════════════════════════════════════════════════════════════════════════════
# so the matrix is going to look like this:
# Each row is a user
# Each column is a game
# And every cell holds a vallue of 1 = liked, -1 = disliked or missing = no review

# A sparse matrix is used because otherwise we are going to get memory issues again? anyway since a lot of users have only reviewed a view of the games
# this type of matrix only stores not zero values. 

#once again just to show it is working on making the matrix aka which step it is at
print("Building user-app sparse matrix ")

# ---So like this part can be removed if we want to just look at the positive reviews and ignore the negative ones.
# Convert voted_up (1/0) to explicit +1 / -1 ratings.
# This ensures thumbs-down reviews are stored as -1 (non-zero) in the sparse
# matrix, rather than 0 which would make them invisible and indistinguishable
# from "never reviewed".
# Also same reason for the .loc[:, ""]
df_filtered.loc[:, "rating"] = df_filtered["voted_up"].map({1: 1, 0: -1})

#--- end removeable code

# csr_matrix takes three things:
# (values, (row_indices, col_indices))  and  shape
# Each review row contributes one non-zero cell: +1 (liked) or -1 (disliked). Cells that are absent entirely (true zeros) mean "never reviewed".
# Also I used csr_matrix instead of the coo_matrix the dude used because. Appently coo is faster to build but slower to do things row by row?
# And that is basically how the KNN stuff works so this works faster for that.
user_app_matrix = csr_matrix(
    (df_filtered["rating"].values,            # +1 or -1 — never 0
     (df_filtered["user_idx"].values,         # which row (user)
      df_filtered["app_idx"].values)),         # which column (app)
    shape=(n_users, n_apps)
)

#Then the fit the model part 
# so metric = cosine means similarity is based on angle between vectors, not magnitude
# and algoritm = brute means compares against every user
model_knn = NearestNeighbors(metric="cosine", algorithm="brute")
# and then fit the matrix with it. Apparenlty this makes it answer quicker for like query time?
model_knn.fit(user_app_matrix)

# Time to announce which step we are at ヽ(￣▽￣)ノ
print("KNN model fitted.")



# Time to actually write the KNN code.... YIPPIE
# I grabbed the nearest 5 but we can change the int in the input if we wanna (apparently one is just the user itself that is why 6)
# Also remember user_idx is that cat code version of the user id
def get_similar_users(user_idx: int, n_neighbors: int = 6):
    #I made row it's own thing because in the dudes code it was in the kneighbors thing but I wanted it seprated to see better what is going on where.
    row = user_app_matrix.getrow(user_idx)   # extract this user's row as a sparse vector. (This basically also doesn't store the 0... save the pc's)
    distances, indices = model_knn.kneighbors(row, n_neighbors=n_neighbors)
    similar_idxs = indices.flatten()[1:]     # [1:] skips the user themselves since like they are 0
    # Convert integer indices back to real steam-IDs using our reverse-lookup array. (aka this gives the true ID not the cat one)
    return [unique_user_ids[i] for i in similar_idxs]
    # ^you need the [] around it otherwise it gives you an error.

# a quick test just for the fun. And to see if it works
sample_user_idx = 0
sample_steamid  = unique_user_ids[sample_user_idx]   # real ID for index 0
similar_users   = get_similar_users(sample_user_idx)
print(f"Similar users to steam-ID {sample_steamid}:")
print(similar_users)
# Yippie it does print things
# things to fix: it now gives np.int64(76561198011792093) as users instead of just the int. HEY FUTURE ME LOOK HERE AND FIX THIS SMALL THINGY


# ═════════════════════════════════════════════════════════════════════════════
# 3. CONTENT-BASED FILTERING  (TF-IDF + Cosine Similarity)
# ═════════════════════════════════════════════════════════════════════════════
# Okay so the idea behind this one. Smack the two dataframes togheter on the appids.
# Then make all the tags just one long string
# Let TF-IDF lose on those strings. commons tags like "singleplayer" or something really normal get low weight
# But rarer tags like "shooter" (idk what is rare okay so bare with me CS:GO was the first game in the list) get a high weight.
# This makes games with similiar high weight tags closer to eachother then if they would be sharing a low weight one.
# And then grab the cosine similarity from all the tag vectors pairs. The more game overlap in tags the closer to 1 they get.




Unique users : 652,993
Unique apps  : 97,056
Building user-app sparse matrix 
KNN model fitted.
Similar users to steam-ID 76561197960267615:
[np.int64(76561198011792093), np.int64(76561198187248634), np.int64(76561197960284066), np.int64(76561197999711370), np.int64(76561198028748141)]


## I dont know, some social justice warrior stuff